# Manual Neuron Annotation Tool

This notebook allows you to interactively view neuron meshes and assign labels to them (e.g., 'exc', 'inh').
Annotations are saved to `annotations.csv`.

**Note**: `annotations.csv` will be created automatically after you save your first annotation.

In [11]:
import os
import pandas as pd
import numpy as np
import caveclient
from meshparty import trimesh_io, trimesh_vtk
from tqdm.notebook import tqdm
import traceback

# Constants
DATASTACK = 'minnie65_public'
ANNOTATION_FILE = 'annotations.csv'

# Initialize Client
client = caveclient.CAVEclient(DATASTACK)
print(f"Connected to datastack: {DATASTACK}")

In [12]:
def load_existing_annotations(filepath):
    if os.path.exists(filepath):
        return pd.read_csv(filepath)
    else:
        return pd.DataFrame(columns=['root_id', 'label', 'notes'])

def save_annotation(root_id, label, notes, filepath):
    df = load_existing_annotations(filepath)
    # Remove existing entry for this root_id if present to update it
    df = df[df['root_id'] != root_id]
    
    new_row = pd.DataFrame([{'root_id': root_id, 'label': label, 'notes': notes}])
    df = pd.concat([df, new_row], ignore_index=True)
    df.to_csv(filepath, index=False)
    print(f"✔️ Saved annotation for {root_id}: {label}")

def get_candidate_root_ids(client, limit=10):
    # Fetch a sample of neurons from the nucleus table
    # Modifying criteria to get a good mix (e.g. limit by volume or location if needed)
    # Note: pt_root_id can sometimes be 0 or invalid in early detections, good to filter.
    nuc_df = client.materialize.query_table('nucleus_detection_v0', limit=limit)
    ids = nuc_df['pt_root_id'].unique().tolist()
    return [x for x in ids if x != 0]

In [13]:
# Setup CloudVolume for mesh downloading
cvol = client.info.segmentation_cloudvolume()
cvol.progress = False  # Disable verbose cloudvolume progress bars

## Visual Annotation Loop

Run the cell below to start annotating. 
- It will pop up a VTK window showing the neuron mesh.
- Close the window to proceed to the input prompt.
- Enter the label (e.g., `exc`, `inh`) or `skip` to skip.
- Type `exit` to stop the loop.

In [14]:
candidates = get_candidate_root_ids(client, limit=5)
existing_df = load_existing_annotations(ANNOTATION_FILE)
processed_ids = set(existing_df['root_id'].values)

print(f"Found {len(candidates)} candidates. {len(processed_ids)} already annotated.")

# Initialized with explicit paths to avoid NoneType errors
mm = trimesh_io.MeshMeta(cv_path=client.info.segmentation_source(), disk_cache_path='meshes')

for root_id in candidates:
    if root_id == 0:
        continue
    if root_id in processed_ids:
        continue
        
    print(f"\nProcessing Root ID: {root_id}")
    
    try:
        # Download Mesh
        # Use seg_id keyword arg because positional arg is treated as filename string
        mesh = mm.mesh(seg_id=root_id)
        
        # check if mesh is valid
        if mesh is None or mesh.vertices.shape[0] == 0:
            print(f"⚠️ Mesh for {root_id} is empty or None. Skipping.")
            continue

        # Visualize
        # Using trimesh_vtk for interactive viewer (opens separate window)
        print("Opening visualization window... Close window to continue.")
        
        # Create a simple actor and render
        actor = trimesh_vtk.mesh_actor(mesh, color=(0.8, 0.2, 0.2))
        trimesh_vtk.render_actors([actor])
        
        # Prompt user
        user_input = input(f"Label for {root_id} (exc/inh/skip/exit): ").strip().lower()
        
        if user_input == 'exit':
            print("Exiting annotation loop.")
            break
        elif user_input == 'skip':
            print("Skipping...")
            continue
        elif user_input:
            save_annotation(root_id, user_input, "", ANNOTATION_FILE)
            
    except Exception as e:
        print(f"❌ Error fetching/visualizing {root_id}: {e}")
        # Common error: Shard configuration missing for specific LoD
        if "shard configuration" in str(e):
             print("   (This usually means the object is too small or segmentation level is mismatched.)")
        continue